# Dijet JEU systematic uncertainty

Compare the JEU Up, Down, and default reconstructed dijet pseudorapidity distributions in MinimumBias, Jet60, Jet80, and Jet100 data. Full CM distributions are normalized to unit integral before comparison. Forward/Backward distributions are left unnormalized; their ratios always use standard independent-error propagation, never ROOT's binomial option.

Dedicated variation/default plots provide the signed shape variations used to estimate the JEU systematic uncertainty.

In [1]:
%load_ext autoreload
%autoreload 2

from dataclasses import replace
from pathlib import Path
import math
import os
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / 'hist_analysis').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Run this notebook from the jetAnalysis repository root')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    import ROOT
except ModuleNotFoundError:
    for path in (Path('/opt/homebrew/lib/python3.14/site-packages'),
                 Path('/opt/homebrew/Cellar/root/6.40.02_1/lib/root')):
        if path.exists() and str(path) not in sys.path:
            sys.path.insert(0, str(path))
    import ROOT

ROOT.gROOT.SetBatch(True)
ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetPalette(ROOT.kBird)
ROOT.TH1.AddDirectory(False)

from hist_analysis.config.files import BASE_DIR
from hist_analysis.config.histograms import (
    DIJET_DELTA_PHI_SELECTION_LABEL,
    STANDARD_DIJET_ETA_CUT_INDEX,
)
from hist_analysis.python.dijet_closures import (
    DijetClosureCurve, build_dijet_gen_comparisons,
)
from hist_analysis.python.histogram_ops import ratio_to_nominal
from hist_analysis.python.plotting import draw_overlay
from hist_analysis.python.root_style import COLORS, DEFAULT_PLOT_STYLE

## Configuration

`PTAVE_BINS` independently selects the intervals analyzed for each trigger sample, following `06_data_check.ipynb`. `FULL_COMPARISON_RATIO_OPTION` controls errors on normalized Up/Def and Down/Def shape ratios. `FB_COMPARISON_RATIO_OPTION` controls errors only on the later ratio of two already-constructed F/B histograms. Set either to `''` for ROOT's standard propagation or `'B'` for option B. The construction of every F/B histogram is hard-coded to `''`.

In [2]:
DATA_DIR = Path(os.environ.get('PPB_DATA_DIR', BASE_DIR / 'exp'))
DATA_FILES = {
    'MinimumBias': DATA_DIR / 'mb_ak4_jetId.root',
    'Jet60': DATA_DIR / 'jet60_ak4_jetId.root',
    'Jet80': DATA_DIR / 'jet80_ak4_jetId.root',
    'Jet100': DATA_DIR / 'jet100_ak4_jetId.root',
}
PTAVE_BINS = {
    'MinimumBias': [(60, 80), (80, 100), (100, 120), (120, 180)],
    'Jet60': [(80, 100), (100, 120), (120, 180), (180, 250)],
    'Jet80': [(100, 120), (120, 180), (180, 250), (300, 500)],
    'Jet100': [(120, 180), (180, 250), (300, 500)],
}
ETA_CUTS = (1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.5)
ETA_CUT_INDEX = STANDARD_DIJET_ETA_CUT_INDEX
REBIN_ETA = 2
FORWARD_BACKWARD_RATIO_OPTION = ''  # protected: F / B is never binomial
FULL_COMPARISON_RATIO_OPTION = ''   # '' or 'B' for Up/Def and Down/Def
FB_COMPARISON_RATIO_OPTION = ''     # '' or 'B' for (F/B)_var / (F/B)_Def
FULL_RATIO_RANGE = (0.85, 1.15)
FB_RANGE = (0.75, 1.30)
FB_DOUBLE_RATIO_RANGE = (0.85, 1.15)
DRAW_GRID = True
SAVE_PNG = False
OUTPUT_DIR = Path(os.environ.get(
    'DIJET_JEU_SYSTEMATICS_OUTPUT_DIR',
    PROJECT_ROOT / 'hist_analysis' / 'output' / 'systematics_JEU',
))

CURVES = (
    DijetClosureCurve(
        'JEU Up', 'hRecoDijetPtEtaCMJeuUp_{eta_cut_index}',
        'hRecoDijetPtEtaForwardJeuUp_{eta_cut_index}',
        'hRecoDijetPtEtaBackwardJeuUp_{eta_cut_index}',
    ),
    DijetClosureCurve(
        'JEU Down', 'hRecoDijetPtEtaCMJeuDown_{eta_cut_index}',
        'hRecoDijetPtEtaForwardJeuDown_{eta_cut_index}',
        'hRecoDijetPtEtaBackwardJeuDown_{eta_cut_index}',
    ),
    DijetClosureCurve(
        'JEU Default', 'hRecoDijetPtEtaCM_{eta_cut_index}',
        'hRecoDijetPtEtaForward_{eta_cut_index}',
        'hRecoDijetPtEtaBackward_{eta_cut_index}',
    ),
)
NOMINAL = 'JEU Default'
# Select entries from root_style.COLORS (0 red, 1 blue, 2 black, ...).
HISTOGRAM_COLOR_INDICES = {'JEU Up': 0, 'JEU Down': 1, 'JEU Default': 2}
VARIATION_COLOR_INDICES = {'Up / Def': 0, 'Down / Def': 1}
PLOT_STYLE = replace(
    DEFAULT_PLOT_STYLE,
    annotation_text_size=0.026, annotation_line_spacing=0.039,
    legend_text_size=0.028,
)

if set(PTAVE_BINS) != set(DATA_FILES):
    raise ValueError('PTAVE_BINS must define exactly the same samples as DATA_FILES')
for sample, intervals in PTAVE_BINS.items():
    if not intervals:
        raise ValueError(f'PTAVE_BINS[{sample!r}] must not be empty')
    if any(low >= high for low, high in intervals):
        raise ValueError(f'Invalid pTave interval for {sample}: {intervals}')
for sample, filename in DATA_FILES.items():
    if not filename.exists():
        raise FileNotFoundError(f'Missing configured {sample} ROOT file: {filename}')
if ETA_CUT_INDEX < 0 or ETA_CUT_INDEX >= len(ETA_CUTS):
    raise IndexError(f'Invalid eta-cut index: {ETA_CUT_INDEX}')
if FORWARD_BACKWARD_RATIO_OPTION != '':
    raise ValueError('Forward/Backward construction must use standard errors')
for option_name, option in (
    ('FULL_COMPARISON_RATIO_OPTION', FULL_COMPARISON_RATIO_OPTION),
    ('FB_COMPARISON_RATIO_OPTION', FB_COMPARISON_RATIO_OPTION),
):
    if option not in ('', 'B'):
        raise ValueError(f'{option_name} must be empty or B')
for label, color_index in (
    *HISTOGRAM_COLOR_INDICES.items(), *VARIATION_COLOR_INDICES.items(),
):
    if not isinstance(color_index, int) or not 0 <= color_index < len(COLORS):
        raise ValueError(f'Invalid root_style color index for {label}: {color_index}')
ETA_CUT = ETA_CUTS[ETA_CUT_INDEX]
DATA_FILES, PTAVE_BINS


({'MinimumBias': PosixPath('/Users/gnigmat/cernbox/ana/pPb8160/exp/mb_ak4_jetId.root'),
  'Jet60': PosixPath('/Users/gnigmat/cernbox/ana/pPb8160/exp/jet60_ak4_jetId.root'),
  'Jet80': PosixPath('/Users/gnigmat/cernbox/ana/pPb8160/exp/jet80_ak4_jetId.root'),
  'Jet100': PosixPath('/Users/gnigmat/cernbox/ana/pPb8160/exp/jet100_ak4_jetId.root')},
 {'MinimumBias': [(60, 80), (80, 100), (100, 120), (120, 180)],
  'Jet60': [(80, 100), (100, 120), (120, 180), (180, 250)],
  'Jet80': [(100, 120), (120, 180), (180, 250), (300, 500)],
  'Jet100': [(120, 180), (180, 250), (300, 500)]})

## Build projections and systematic comparisons

Each CM projection is scaled by `1 / Integral()` through `normalization='integral'`. Forward and backward projections are not normalized before division. The shared helper is called separately below for every trigger sample.

In [3]:
def finite_nonzero_range(histogram):
    values = [
        histogram.GetBinContent(index)
        for index in range(1, histogram.GetNbinsX() + 1)
        if histogram.GetBinContent(index) != 0.0
        and math.isfinite(histogram.GetBinContent(index))
    ]
    return (min(values), max(values)) if values else None


def analyze_data_file(sample, input_file):
    results = {}
    eta_x_range = (-ETA_CUT - 0.1, ETA_CUT + 0.1)
    fb_x_range = (0.0, ETA_CUT + 0.1)
    eta_cut_tag = int(round(10.0 * ETA_CUT))
    sample_tag = sample.lower()

    for ptave_range in PTAVE_BINS[sample]:
        low, high = ptave_range
        ptave_tag = f'{low:g}_{high:g}'.replace('.', 'p')
        common_tag = (
            f'{sample_tag}_jeuSystematics_etaCM_{eta_cut_tag}'
            f'_ptave_{ptave_tag}'
        )
        output_name = lambda plot: (
            f'{sample_tag}_jeuSystematics_{plot}'
            f'_etaCM_{eta_cut_tag}_ptave_{ptave_tag}.pdf'
        )
        eta_shapes, fb_ratios, selected_keys = build_dijet_gen_comparisons(
            input_file, CURVES, eta_cut_index=ETA_CUT_INDEX,
            ptave_range=ptave_range, nominal=NOMINAL, rebin_eta=REBIN_ETA,
            normalization='integral',
            ratio_option=FORWARD_BACKWARD_RATIO_OPTION,
        )
        eta_variations = {
            ratio_label: ratio_to_nominal(
                eta_shapes[source_label], eta_shapes[NOMINAL],
                name=f'h_{common_tag}_{source_label.replace(" ", "_")}_to_default',
                option=FULL_COMPARISON_RATIO_OPTION,
            )
            for ratio_label, source_label in (
                ('Up / Def', 'JEU Up'), ('Down / Def', 'JEU Down'),
            )
        }
        fb_variations = {
            ratio_label: ratio_to_nominal(
                fb_ratios[source_label], fb_ratios[NOMINAL],
                name=f'h_{common_tag}_{source_label.replace(" ", "_")}_fb_to_default',
                option=FB_COMPARISON_RATIO_OPTION,
            )
            for ratio_label, source_label in (
                ('Up / Def', 'JEU Up'), ('Down / Def', 'JEU Down'),
            )
        }
        annotations = (
            sample,
            f'{low:g} < p_{{T}}^{{ave}} < {high:g} GeV',
            f'|#eta_{{CM}}^{{jet}}| < {ETA_CUT:g}',
            'p_{T}^{Lead} > 50 GeV',
            'p_{T}^{SubLead} > 40 GeV',
            DIJET_DELTA_PHI_SELECTION_LABEL,
        )
        canvases = {
            'eta_overlay': draw_overlay(
                eta_shapes, title='', x_title='#eta_{CM}^{dijet}',
                y_title='1/N dN/d#eta_{CM}^{dijet}', x_range=eta_x_range,
                annotations=annotations, grid=DRAW_GRID, headroom=1.6,
                style_indices=HISTOGRAM_COLOR_INDICES, style=PLOT_STYLE,
                output=OUTPUT_DIR / output_name('full_overlay'),
                save_png=SAVE_PNG, canvas_name=f'{common_tag}_full_overlay',
            ),
            'eta_variations': draw_overlay(
                eta_variations, title='', x_title='#eta_{CM}^{dijet}',
                y_title='JEU variation / default', x_range=eta_x_range,
                y_range=FULL_RATIO_RANGE, reference_y=1.0,
                annotations=annotations, grid=DRAW_GRID,
                style_indices=VARIATION_COLOR_INDICES, style=PLOT_STYLE,
                output=OUTPUT_DIR / output_name('full_ratio_to_default'),
                save_png=SAVE_PNG, canvas_name=f'{common_tag}_full_ratio',
            ),
            'fb_overlay': draw_overlay(
                fb_ratios, title='', x_title='#eta_{CM}^{dijet}',
                y_title='Forward / Backward', x_range=fb_x_range,
                y_range=FB_RANGE, annotations=annotations, grid=DRAW_GRID,
                style_indices=HISTOGRAM_COLOR_INDICES, style=PLOT_STYLE,
                output=OUTPUT_DIR / output_name('fb_overlay'),
                save_png=SAVE_PNG, canvas_name=f'{common_tag}_fb_overlay',
            ),
            'fb_variations': draw_overlay(
                fb_variations, title='', x_title='#eta_{CM}^{dijet}',
                y_title='(F/B)_{JEU variation} / (F/B)_{default}',
                x_range=fb_x_range, y_range=FB_DOUBLE_RATIO_RANGE,
                reference_y=1.0, annotations=annotations, grid=DRAW_GRID,
                style_indices=VARIATION_COLOR_INDICES, style=PLOT_STYLE,
                output=OUTPUT_DIR / output_name('fb_ratio_to_default'),
                save_png=SAVE_PNG, canvas_name=f'{common_tag}_fb_ratio',
            ),
        }
        results[ptave_range] = {
            'eta_shapes': eta_shapes, 'eta_variations': eta_variations,
            'forward_backward': fb_ratios, 'fb_variations': fb_variations,
            'keys': selected_keys, 'canvases': canvases,
        }
        print(f'\n{sample}, pTave interval {ptave_range}, eta cut {ETA_CUT:g}')
        print('selected keys:', selected_keys)
        print('CM integral normalization:', {
            label: histogram.Integral()
            for label, histogram in eta_shapes.items()
        })
        print('CM variation/default ranges:', {
            label: finite_nonzero_range(histogram)
            for label, histogram in eta_variations.items()
        })
        print('F/B variation/default ranges:', {
            label: finite_nonzero_range(histogram)
            for label, histogram in fb_variations.items()
        })
        for canvas in canvases.values():
            display(canvas)
    return results


## MinimumBias data

In [4]:
mb_results = analyze_data_file('MinimumBias', DATA_FILES['MinimumBias'])


ValueError: Cannot normalize 'dijet_closure_cm_0': integral is 0.0

## Jet60 data

In [ ]:
jet60_results = analyze_data_file('Jet60', DATA_FILES['Jet60'])


## Jet80 data

In [ ]:
jet80_results = analyze_data_file('Jet80', DATA_FILES['Jet80'])


## Jet100 data

In [ ]:
jet100_results = analyze_data_file('Jet100', DATA_FILES['Jet100'])
